# Week 4 — Classification Layer

This notebook trains classifiers to predict **ticket category** and **priority** from the description text.

**Plan:**
| Model | Target | Notes |
|---|---|---|
| TF-IDF + Logistic Regression | category | Baseline — spoiler: synthetic data leakage |
| TF-IDF + Logistic Regression | priority | Real baseline — 38% F1, weak |
| DistilBERT | priority | Transformer fine-tune — meaningful improvement |

All runs logged to **MLflow**. Start the UI with `mlflow ui` in the terminal.

## 1. Setup

In [ ]:
import sys, os

# Only move up if we're running from the notebooks/ directory.
# This makes the cell safe to re-run without drifting further up the tree.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.insert(0, os.getcwd())

import pandas as pd
import matplotlib.pyplot as plt
from src.classifier import load_data, train_tfidf_lr

print(f"Working directory: {os.getcwd()}")
print("Imports OK")

## 2. Data — class distribution

Understanding class balance before training matters.
If one class dominates, a dumb model that always predicts that class gets high accuracy for free.
We use **weighted F1** as the primary metric to account for this.

In [ ]:
df = pd.read_csv("data/processed/tickets_clean.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df["category"].value_counts().plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Category distribution")
axes[0].set_xlabel("Count")

df["priority"].value_counts().plot(kind="barh", ax=axes[1], color="coral")
axes[1].set_title("Priority distribution")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

print("Priority — majority class baseline (always predict Medium):")
majority = df["priority"].value_counts(normalize=True).max()
print(f"  Accuracy if we predict 'Medium' every time: {majority:.1%}")
print("  Our TF-IDF baseline needs to beat this to be useful.")

## 3. Load data

In [ ]:
X_train, X_test, yc_train, yc_test, yp_train, yp_test = load_data()
print(f"Train: {len(X_train):,} tickets")
print(f"Test:  {len(X_test):,} tickets")
print(f"\nExample input: {X_train[0][:200]}")

## 4. Baseline — TF-IDF + LR on Category

Expected result: suspiciously high accuracy.
The synthetic dataset's templates embedded category-specific words directly into descriptions
(e.g. the word "database" appears in 83% of Database ticket descriptions).
TF-IDF picks this up trivially — it's not generalisation, it's memorisation.

> **This is a real lesson:** always sanity-check your data for leakage before trusting a high accuracy score.

In [ ]:
cat_results = train_tfidf_lr(
    X_train, X_test, yc_train, yc_test,
    target="category"
)

## 5. Baseline — TF-IDF + LR on Priority

Priority is the real challenge. There are no obvious keyword shortcuts —
the model needs to understand urgency and impact from the ticket text.
TF-IDF can't do this well: it sees word counts, not meaning.

In [ ]:
pri_results = train_tfidf_lr(
    X_train, X_test, yp_train, yp_test,
    target="priority"
)

## 6. Open MLflow UI

In a terminal, run:
```bash
mlflow ui
```
Then open http://localhost:5000

You should see two runs: `tfidf_lr_category` and `tfidf_lr_priority`.
Each run shows the parameters (C, max_features) and metrics (accuracy, F1).
This is the dashboard that gets better as you add more runs below.

## 7. DistilBERT — Priority Classifier

DistilBERT is a smaller, faster version of BERT — 40% fewer parameters, 60% faster,
retains 97% of BERT's performance. It reads the full ticket description as a sequence
of tokens and learns contextual representations — unlike TF-IDF which just counts words.

This is where we expect to see a real jump in priority F1.

In [ ]:
from src.classifier import train_distilbert

distilbert_results = train_distilbert(
    X_train, X_test, yp_train, yp_test,
    target="priority",
    epochs=5,
    lr=3e-5,
)

## 8. Model Comparison

Side-by-side comparison of TF-IDF baseline vs DistilBERT on priority prediction.

In [ ]:
comparison = {
    "TF-IDF + LR": pri_results,
    "DistilBERT": distilbert_results,
}

print(f"{'Model':<20} {'Accuracy':>10} {'F1 (weighted)':>15}")
print("-" * 47)
for name, r in comparison.items():
    print(f"{name:<20} {r['accuracy']:>10.4f} {r['f1_weighted']:>15.4f}")

improvement = distilbert_results['f1_weighted'] - pri_results['f1_weighted']
print(f"\nF1 improvement from TF-IDF → DistilBERT: +{improvement:.4f}")

## Summary

| Model | Target | Accuracy | F1 (weighted) | Notes |
|---|---|---|---|---|
| TF-IDF + LR | category | ~1.0 | ~1.0 | Data leakage — category keywords in descriptions |
| TF-IDF + LR | priority (original data) | ~0.38 | ~0.28 | Priority assigned randomly — not learnable |
| TF-IDF + LR | priority (fixed data) | ~0.88 | ~0.88 | Explicit urgency keywords — TF-IDF handles well |
| DistilBERT | priority (fixed data) | ~0.87 | ~0.87 | Comparable to TF-IDF on keyword-based signal |

**Key takeaway:** The biggest performance jump came from fixing the data (28% → 87%), not from switching models. TF-IDF matches DistilBERT here because the priority signal is explicit keyword-based urgency language — transformers show their advantage when the signal is subtle and contextual. This is the "garbage in, garbage out" principle in action.

**Week 5:** Wrap the RAG chain + classifier into a Flask API with `/classify` and `/resolve` endpoints, then build a Streamlit UI.